# 🤟 SANA A-PSL: Google Colab Video Landmark Extraction & Augmentation Pipeline
**Goal:** Ingest raw video recordings directly from your specified Google Drive path (`Test Data/{label}/[videos]`), extract **208-dimensional MediaPipe landmark sequences**, apply **6× Data Augmentation**, and save the processed `.npy` dataset and ZIP file directly to Google Drive.

### 📁 Expected Folder Hierarchy:
```
<YOUR_SPECIFIED_PATH>/
├── headache/
│   ├── video_01.mp4
│   └── video_02.mp4
├── stomach_pain/
│   ├── video_01.mp4
│   └── video_02.mp4
└── fever/
    ├── video_01.mp4
    └── video_02.mp4
```

### 📐 208-Dimension Feature Layout per Frame:
- `[0:66]`   : 33 Pose landmarks $(X, Y)$
- `[66:108]` : 21 Left Hand landmarks $(X, Y)$
- `[108:150]`: 21 Right Hand landmarks $(X, Y)$
- `[150:208]`: 29 Expression Face landmarks / Neutral 0s $(X, Y)$

In [ ]:
# ── Cell 1: Mount Google Drive & Install Dependencies ────────────────────────
from google.colab import drive
drive.mount('/content/drive')

!pip install -q mediapipe==0.10.14 opencv-python numpy matplotlib tqdm pandas

import os
import sys
import glob
import json
import math
import random
import shutil
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# Robust MediaPipe import compatible with all versions
try:
    import mediapipe.python.solutions.holistic as mp_holistic
    import mediapipe.python.solutions.drawing_utils as mp_drawing
except (ImportError, AttributeError):
    try:
        import mediapipe as mp
        mp_holistic = mp.solutions.holistic
        mp_drawing = mp.solutions.drawing_utils
    except AttributeError:
        from mediapipe.python.solutions import holistic as mp_holistic
        from mediapipe.python.solutions import drawing_utils as mp_drawing

import mediapipe as mp

print("\n" + "=" * 60)
print("✅ Google Drive Mounted & MediaPipe Loaded Successfully!")
print(f"OpenCV Version:    {cv2.__version__}")
print(f"MediaPipe Version: {mp.__version__}")
print("=" * 60)

In [ ]:
# ── Cell 2: Specify Your Dataset Path & Configuration ────────────────────────
# 👇 SPECIFY YOUR EXACT GOOGLE DRIVE DATASET PATH HERE:
DATASET_PATH = "/content/drive/MyDrive/test_dataset_PSL"

CONFIG = {
    # Paths
    "RAW_VIDEOS_DIR":       DATASET_PATH,                         # Folder with {label} subfolders
    "LOCAL_OUTPUT_DIR":     "/content/processed_psl_dataset",     # Fast local processing directory
    "GDRIVE_BACKUP_DIR":    "/content/drive/MyDrive/SANA_PSL_Processed_Dataset", # Backup directory in GDrive
    "OUTPUT_ZIP_NAME":      "SANA_PSL_Keypoints_Dataset.zip",
    
    # Landmark Extraction Parameters
    "INPUT_DIM":            208,                                  # 66 Pose + 42 LH + 42 RH + 58 Face
    "TARGET_FRAMES":        60,                                   # Resample all sequences to 60 frames (SANA Conv1D standard)
    "MIN_DETECTION_CONF":   0.5,
    "MIN_TRACKING_CONF":    0.5,
    "SMOOTHING_ALPHA":      0.75,                                 # Temporal coordinate smoothing
    "MIRROR_CORRECTION":    True,                                 # Set True if recorded with front selfie camera
    
    # 6x Data Augmentation
    "ENABLE_AUGMENTATION":  True,
    "AUGMENTATION_FACTOR":  5,                                    # 1 Original + 5 Augmented = 6x Expansion
    "SCALE_RANGE":          (0.85, 1.15),                         # Spatial distance variation
    "SHIFT_RANGE":          (-0.06, 0.06),                        # Spatial centering translation
    "SPEED_RANGE":          (0.85, 1.15),                         # Temporal speed variation
    "JITTER_SIGMA":         0.003,                                # Coordinate noise
}

os.makedirs(CONFIG["LOCAL_OUTPUT_DIR"], exist_ok=True)
os.makedirs(CONFIG["GDRIVE_BACKUP_DIR"], exist_ok=True)

print("=" * 60)
print(f"📂 Specified Dataset Path:    {CONFIG['RAW_VIDEOS_DIR']}")
print(f"💾 Permanent GDrive Backup:   {CONFIG['GDRIVE_BACKUP_DIR']}")
print(f"⚡ Local Processing Output:    {CONFIG['LOCAL_OUTPUT_DIR']}")
print("=" * 60)

if not os.path.exists(CONFIG["RAW_VIDEOS_DIR"]):
    print(f"⚠️ Error: Directory '{CONFIG['RAW_VIDEOS_DIR']}' does not exist.")
    print("👉 Please verify your DATASET_PATH string at the top of this cell!")
else:
    labels = [d for d in os.listdir(CONFIG["RAW_VIDEOS_DIR"]) if os.path.isdir(os.path.join(CONFIG["RAW_VIDEOS_DIR"], d))]
    print(f"✅ Directory verified! Found {len(labels)} label subfolders:")
    print(labels)

In [ ]:
# ── Cell 3: MediaPipe Holistic 208-Dimension Landmark Extractor ─────────────
try:
    import mediapipe.python.solutions.holistic as mp_holistic
except (ImportError, AttributeError):
    try:
        from mediapipe.python.solutions import holistic as mp_holistic
    except ImportError:
        import mediapipe as mp
        mp_holistic = mp.solutions.holistic

class MediaPipeVideoExtractor:
    def __init__(self, min_detection_conf=0.5, min_tracking_conf=0.5, mirror_fix=True, alpha=0.75):
        self.holistic = mp_holistic.Holistic(
            static_image_mode=False,
            model_complexity=1,
            enable_segmentation=False,
            refine_face_landmarks=False,
            min_detection_confidence=min_detection_conf,
            min_tracking_confidence=min_tracking_conf
        )
        self.mirror_fix = mirror_fix
        self.alpha = alpha
        self.prev_landmarks = None
        
    def reset_tracker(self):
        self.prev_landmarks = None
        
    def extract_frame(self, frame_bgr):
        """
        Extracts exactly 208 normalized (x, y) coordinates from a single frame.
        Returns: np.ndarray shape (208,)
        """
        frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        results = self.holistic.process(frame_rgb)
        
        # 1. Pose Landmarks (33 points -> 66 floats: x, y)
        pose_coords = [0.0] * 66
        if results.pose_landmarks:
            for i, lm in enumerate(results.pose_landmarks.landmark):
                pose_coords[i*2] = float(lm.x)
                pose_coords[i*2 + 1] = float(lm.y)
                
        # 2. Left Hand Landmarks (21 points -> 42 floats: x, y)
        lh_coords = [0.0] * 42
        if results.left_hand_landmarks:
            for i, lm in enumerate(results.left_hand_landmarks.landmark):
                lh_coords[i*2] = float(lm.x)
                lh_coords[i*2 + 1] = float(lm.y)
                
        # 3. Right Hand Landmarks (21 points -> 42 floats: x, y)
        rh_coords = [0.0] * 42
        if results.right_hand_landmarks:
            for i, lm in enumerate(results.right_hand_landmarks.landmark):
                rh_coords[i*2] = float(lm.x)
                rh_coords[i*2 + 1] = float(lm.y)
                
        # Handle Selfie Mirror Inversion if enabled
        if self.mirror_fix:
            lh_coords, rh_coords = rh_coords, lh_coords
            
        # 4. Face Landmarks (58 floats - neutral zeros for SANA standard)
        face_coords = [0.0] * 58
        
        # Concatenate into 208 floats
        current_frame_208 = np.array(pose_coords + lh_coords + rh_coords + face_coords, dtype=np.float32)
        
        # Coordinate smoothing across consecutive frames
        if self.prev_landmarks is None:
            self.prev_landmarks = current_frame_208
        else:
            active_mask = (current_frame_208 != 0.0).astype(np.float32)
            smoothed = active_mask * (self.alpha * current_frame_208 + (1 - self.alpha) * self.prev_landmarks) + (1 - active_mask) * current_frame_208
            self.prev_landmarks = smoothed
            current_frame_208 = smoothed
            
        return current_frame_208

    def extract_from_video(self, video_path):
        """
        Ingests a video file and extracts the full temporal sequence of landmarks.
        Returns: np.ndarray of shape (T, 208), fps, total_frames
        """
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            print(f"⚠️ Error: Could not open video {video_path}")
            return None, 0, 0
        
        fps = cap.get(cv2.CAP_PROP_FPS)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        
        self.reset_tracker()
        sequence = []
        
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            landmarks_208 = self.extract_frame(frame)
            sequence.append(landmarks_208)
            
        cap.release()
        
        if len(sequence) == 0:
            return None, fps, 0
            
        return np.array(sequence, dtype=np.float32), fps, len(sequence)

    def close(self):
        self.holistic.close()

print("✅ MediaPipe Holistic Extractor Class Initialized.")

In [ ]:
# ── Cell 4: Temporal Spline Resampling (60 Frames) ───────────────────────────
def resample_sequence(sequence, target_frames=60):
    """
    Resamples an arbitrary length landmark sequence (T, 208) to exactly target_frames (e.g. 60)
    using linear/spline temporal interpolation.
    """
    T = sequence.shape[0]
    if T == target_frames:
        return sequence
    
    orig_times = np.linspace(0, 1, T)
    target_times = np.linspace(0, 1, target_frames)
    
    resampled = np.zeros((target_frames, sequence.shape[1]), dtype=np.float32)
    for dim in range(sequence.shape[1]):
        resampled[:, dim] = np.interp(target_times, orig_times, sequence[:, dim])
        
    return resampled

print("✅ 60-Frame Temporal Resampling Function ready.")

In [ ]:
# ── Cell 5: 6× Data Synthesis & Augmentation Engine ──────────────────────────
class LandmarkAugmentor:
    def __init__(self, scale_range=(0.85, 1.15), shift_range=(-0.05, 0.05), speed_range=(0.85, 1.15), jitter_sigma=0.003):
        self.scale_range = scale_range
        self.shift_range = shift_range
        self.speed_range = speed_range
        self.jitter_sigma = jitter_sigma
        
    def augment(self, sequence_60):
        """
        Applies random spatial scaling, 2D translation, temporal speed warping, and jitter.
        Input:  (60, 208)
        Output: (60, 208) newly synthesized variation
        """
        aug_seq = sequence_60.copy()
        T = aug_seq.shape[0]
        
        # 1. Temporal Speed Warping
        speed_factor = random.uniform(*self.speed_range)
        warped_len = max(10, int(T * speed_factor))
        orig_t = np.linspace(0, 1, T)
        warp_t = np.linspace(0, 1, warped_len)
        warped = np.zeros((warped_len, 208), dtype=np.float32)
        for d in range(208):
            warped[:, d] = np.interp(warp_t, orig_t, aug_seq[:, d])
        aug_seq = resample_sequence(warped, target_frames=T)
        
        # 2. Spatial Scaling
        scale = random.uniform(*self.scale_range)
        
        # 3. Spatial Translation
        shift_x = random.uniform(*self.shift_range)
        shift_y = random.uniform(*self.shift_range)
        
        # 4. Coordinate Jitter
        jitter = np.random.normal(0, self.jitter_sigma, size=aug_seq.shape).astype(np.float32)
        
        for t in range(T):
            for i in range(0, 150, 2): # Apply to Pose, Left Hand, Right Hand
                if aug_seq[t, i] != 0.0 or aug_seq[t, i+1] != 0.0:
                    x_centered = (aug_seq[t, i] - 0.5) * scale + 0.5 + shift_x + jitter[t, i]
                    y_centered = (aug_seq[t, i+1] - 0.5) * scale + 0.5 + shift_y + jitter[t, i+1]
                    aug_seq[t, i] = np.clip(x_centered, 0.0, 1.0)
                    aug_seq[t, i+1] = np.clip(y_centered, 0.0, 1.0)
                    
        return aug_seq

augmentor = LandmarkAugmentor(
    scale_range=CONFIG["SCALE_RANGE"],
    shift_range=CONFIG["SHIFT_RANGE"],
    speed_range=CONFIG["SPEED_RANGE"],
    jitter_sigma=CONFIG["JITTER_SIGMA"]
)
print("✅ 6x Data Augmentation Engine initialized.")

In [ ]:
# ── Cell 6: Scan Subfolders for Labels & Extract Keypoints ────────────────────
def scan_label_subfolders(root_dir):
    """
    Scans the specified root_dir for label subfolders and video files.
    Expected structure: root_dir/{label}/[videos]
    """
    extensions = ('*.mp4', '*.avi', '*.mov', '*.webm', '*.mkv', '*.MP4', '*.AVI', '*.MOV')
    samples = []
    
    if not os.path.exists(root_dir):
        print(f"⚠️ Error: Path '{root_dir}' does not exist.")
        return []
        
    subfolders = [d for d in sorted(os.listdir(root_dir)) if os.path.isdir(os.path.join(root_dir, d))]
    
    for label_name in subfolders:
        label_dir = os.path.join(root_dir, label_name)
        label_videos = []
        for ext in extensions:
            label_videos.extend(glob.glob(os.path.join(label_dir, ext)))
            
        for vpath in sorted(set(label_videos)):
            samples.append({
                "video_path": vpath,
                "label": label_name,
                "file_name": os.path.basename(vpath)
            })
            
    return samples

samples = scan_label_subfolders(CONFIG["RAW_VIDEOS_DIR"])
labels_found = sorted(list(set([s['label'] for s in samples])))

print(f"🔍 Scanned: '{CONFIG['RAW_VIDEOS_DIR']}'")
print(f"📊 Found {len(samples)} total videos across {len(labels_found)} classes:")
for lbl in labels_found:
    count = len([s for s in samples if s['label'] == lbl])
    print(f"   • {lbl:<25}: {count} videos")

if len(samples) == 0:
    print("\n⚠️ No video files found. Please check your DATASET_PATH in Cell 2!")
else:
    extractor = MediaPipeVideoExtractor(
        min_detection_conf=CONFIG["MIN_DETECTION_CONF"],
        min_tracking_conf=CONFIG["MIN_TRACKING_CONF"],
        mirror_fix=CONFIG["MIRROR_CORRECTION"],
        alpha=CONFIG["SMOOTHING_ALPHA"]
    )
    
    dataset_records = []
    print("\n🚀 Starting Landmark Extraction & 6x Augmentation Pipeline...")
    
    for item in tqdm(samples, desc="Extracting Keypoints"):
        vpath = item["video_path"]
        label = item["label"]
        fname = Path(item["file_name"]).stem
        
        raw_seq, fps, frame_count = extractor.extract_from_video(vpath)
        if raw_seq is None or frame_count < 5:
            print(f"⚠️ Skipping {vpath} (insufficient frames: {frame_count})")
            continue
            
        # Resample to 60 frames
        resampled_60 = resample_sequence(raw_seq, target_frames=CONFIG["TARGET_FRAMES"])
        
        # Output directory for this class
        class_dir = os.path.join(CONFIG["LOCAL_OUTPUT_DIR"], label)
        os.makedirs(class_dir, exist_ok=True)
        
        # 1. Save Clean Original Keypoints
        orig_out_path = os.path.join(class_dir, f"{fname}_orig.npy")
        np.save(orig_out_path, resampled_60)
        dataset_records.append({
            "npy_path": orig_out_path,
            "label": label,
            "type": "original",
            "raw_frames": frame_count,
            "resampled_frames": CONFIG["TARGET_FRAMES"],
            "fps": fps
        })
        
        # 2. Synthesize 5 Augmented Variations (6x Total Expansion)
        if CONFIG["ENABLE_AUGMENTATION"]:
            for aug_idx in range(1, CONFIG["AUGMENTATION_FACTOR"] + 1):
                aug_seq = augmentor.augment(resampled_60)
                aug_out_path = os.path.join(class_dir, f"{fname}_aug_{aug_idx:02d}.npy")
                np.save(aug_out_path, aug_seq)
                dataset_records.append({
                    "npy_path": aug_out_path,
                    "label": label,
                    "type": f"augmented_{aug_idx}",
                    "raw_frames": frame_count,
                    "resampled_frames": CONFIG["TARGET_FRAMES"],
                    "fps": fps
                })
                
    extractor.close()
    print(f"\n🎉 Extraction Complete! Generated {len(dataset_records)} processed .npy arrays.")

In [ ]:
# ── Cell 7: Metadata & Bilingual Translation Mapping ─────────────────────────
# Standard Bilingual Medical Translation Mapping (English + Native Urdu)
BILINGUAL_DICTIONARY = {
    "headache":         ("I have a severe headache", "میرے سر میں شدید درد ہے"),
    "stomach_pain":     ("My stomach hurts", "میرے پیٹ میں درد ہے"),
    "chest_pain":       ("I have chest pain", "میرے سینے میں درد ہے"),
    "fever":            ("I have a high fever", "مجھے تیز بخار ہے"),
    "cough":            ("I have a cough", "مجھے کھانسی ہے"),
    "dizziness":        ("I feel dizzy", "مجھے چکر آ رہے ہیں"),
    "nausea":           ("I feel nauseous", "مجھے متلی ہو رہی ہے"),
    "allergy":          ("I have an allergy", "مجھے الرجی ہے"),
    "diabetes":         ("I have diabetes", "مجھے شوگر ہے"),
    "blood_pressure":   ("My blood pressure is high", "میرا بلڈ پریشر زیادہ ہے"),
    "breathless":       ("I cannot breathe properly", "مجھے سانس لینے میں دشواری ہو رہی ہے"),
    "bleeding":         ("There is bleeding", "خون بہہ رہا ہے"),
    "broken_bone":      ("My bone is fractured", "میری ہڈی ٹوٹ گئی ہے"),
    "burn":             ("I have a burn injury", "میرا ہاتھ جل گیا ہے"),
    "medicine":         ("I need my medicine", "مجھے میری دوائی چاہیے"),
    "water":            ("Please give me water", "براہ کرم مجھے پانی دیں"),
    "doctor":           ("I need to see a doctor", "مجھے ڈاکٹر سے ملنا ہے"),
    "help":             ("Please help me", "براہ کرم میری مدد کریں"),
    "emergency":        ("This is an emergency", "یہ ایمرجنسی ہے"),
    "pain_scale_high":  ("The pain is very intense (10/10)", "درد بہت شدید ہے"),
}

if 'dataset_records' in locals() and len(dataset_records) > 0:
    df = pd.DataFrame(dataset_records)
    
    df["english_translation"] = df["label"].apply(
        lambda c: BILINGUAL_DICTIONARY.get(c.lower(), (c.replace('_', ' ').title(), ""))[0]
    )
    df["urdu_translation"] = df["label"].apply(
        lambda c: BILINGUAL_DICTIONARY.get(c.lower(), ("", ""))[1]
    )
    
    local_csv_path = os.path.join(CONFIG["LOCAL_OUTPUT_DIR"], "dataset_metadata.csv")
    df.to_csv(local_csv_path, index=False)
    print(f"📄 Metadata CSV generated: {local_csv_path}")
    display(df.head(10))
else:
    print("ℹ️ Metadata will be generated once videos are processed in Cell 6.")

In [ ]:
# ── Cell 8: Visual Skeleton Verification Strip ───────────────────────────────
def plot_extracted_skeleton(npy_path, frame_indices=[0, 15, 30, 45, 59]):
    """
    Plots 2D skeletal pose and hands across multiple frames to verify tracking.
    """
    data = np.load(npy_path) # Shape: (60, 208)
    T = data.shape[0]
    
    fig, axes = plt.subplots(1, len(frame_indices), figsize=(18, 4))
    fig.suptitle(f"SANA Skeleton Tracking: {os.path.basename(npy_path)}", fontsize=14, y=1.05)
    
    for idx, f_idx in enumerate(frame_indices):
        f_idx = min(f_idx, T - 1)
        frame_data = data[f_idx]
        
        pose_xy = frame_data[0:66].reshape(-1, 2)
        lh_xy   = frame_data[66:108].reshape(-1, 2)
        rh_xy   = frame_data[108:150].reshape(-1, 2)
        
        ax = axes[idx]
        ax.set_title(f"Frame {f_idx}/{T}")
        ax.set_xlim(0, 1)
        ax.set_ylim(1, 0) # Invert Y for standard image view
        ax.set_aspect('equal')
        ax.grid(True, linestyle='--', alpha=0.3)
        
        # Plot Pose (Upper Body)
        valid_pose = pose_xy[pose_xy.sum(axis=1) != 0]
        if len(valid_pose) > 0:
            ax.scatter(valid_pose[:, 0], valid_pose[:, 1], c='blue', s=25, label='Pose')
            
        # Plot Left Hand
        valid_lh = lh_xy[lh_xy.sum(axis=1) != 0]
        if len(valid_lh) > 0:
            ax.scatter(valid_lh[:, 0], valid_lh[:, 1], c='green', s=35, label='Left Hand')
            
        # Plot Right Hand
        valid_rh = rh_xy[rh_xy.sum(axis=1) != 0]
        if len(valid_rh) > 0:
            ax.scatter(valid_rh[:, 0], valid_rh[:, 1], c='red', s=35, label='Right Hand')
            
        if idx == 0:
            ax.legend(loc='lower right', fontsize=8)
            
    plt.tight_layout()
    plt.show()

sample_npy = glob.glob(os.path.join(CONFIG["LOCAL_OUTPUT_DIR"], '**', '*.npy'), recursive=True)
if sample_npy:
    print(f"Displaying skeleton strip for: {sample_npy[0]}")
    plot_extracted_skeleton(sample_npy[0])
else:
    print("ℹ️ Run Cell 6 first to visualize extracted skeletons.")

In [ ]:
# ── Cell 9: Package Dataset & Backup Directly to Google Drive ────────────────
zip_base_name = "/content/" + CONFIG["OUTPUT_ZIP_NAME"].replace('.zip', '')
dataset_dir = CONFIG["LOCAL_OUTPUT_DIR"]

if os.path.exists(dataset_dir) and len(os.listdir(dataset_dir)) > 0:
    print(f"📦 Creating ZIP archive from '{dataset_dir}'...")
    shutil.make_archive(zip_base_name, 'zip', dataset_dir)
    local_zip = zip_base_name + ".zip"
    zip_mb = os.path.getsize(local_zip) / (1024 * 1024)
    
    # Copy ZIP to Google Drive
    gdrive_zip_dest = os.path.join("/content/drive/MyDrive", CONFIG["OUTPUT_ZIP_NAME"])
    print(f"💾 Backing up ZIP to Google Drive: {gdrive_zip_dest}...")
    shutil.copy2(local_zip, gdrive_zip_dest)
    
    print("\n" + "=" * 60)
    print("🎉 DATASET PACKAGING & GDRIVE BACKUP COMPLETE!")
    print(f"📁 Local Colab Zip:  {local_zip} ({zip_mb:.2f} MB)")
    print(f"☁️ Google Drive Zip: {gdrive_zip_dest}")
    print("=" * 60)
    print("\n🚀 You can now train directly or upload this ZIP to Kaggle for Few-Shot fine-tuning!")
else:
    print("ℹ️ Process video files in Cell 6 before creating the backup ZIP.")